# Basic Agent Demo

Demonstrates ArcAgent capabilities:
1. **Startup** — Identity, tools, extensions, skills
2. **Built-in tools** — ls, read, write, edit, bash, grep, find
3. **Extension tools** — Calculator extension auto-discovered
4. **Self-extension** — Agent creates its own tools at runtime
5. **Skills** — Agent creates and discovers SKILL.md files
6. **Multi-turn chat** — Session persistence and memory
7. **ACE** — Agent writes to policy.md and context.md

In [ ]:
import asyncio
from pathlib import Path

from arcagent.core.agent import ArcAgent
from arcagent.core.config import load_config

# Load config
config_path = Path("arcagent.toml")
config = load_config(config_path)
print(f"Agent: {config.agent.name}")
print(f"Model: {config.llm.model}")
print(f"Workspace: {config.agent.workspace}")

## 1. Agent Startup

Initializes identity, telemetry, bus, tools, skills, extensions.

In [ ]:
agent = ArcAgent(config, config_path=config_path)
await agent.startup()

print(f"DID: {agent._identity.did}")
print(f"Started: {agent._started}")

## 2. Inspect Registered Tools

7 built-in + 1 calculator extension = 8 tools.

In [ ]:
tools = agent._tool_registry.tools
print(f"Total tools: {len(tools)}\n")
for name, tool in sorted(tools.items()):
    print(f"  {name:12s}  [{tool.transport.value:6s}]  {tool.source}")

## 3. Inspect Extensions

The calculator extension was auto-discovered from `workspace/extensions/`.

In [ ]:
manifests = agent._extension_loader.manifests
print(f"Extensions loaded: {len(manifests)}\n")
for m in manifests:
    print(f"  {m.name}: tools={m.tools_registered}, sandbox={m.sandbox_mode}, {m.load_time_ms:.1f}ms")

## 4. Single-Shot Task — Built-in Tools

Agent uses `ls` tool to list workspace files.

In [ ]:
result = await agent.run("List all files in the workspace")
print(result.content)

## 5. Single-Shot Task — Extension Tool

Agent uses the `calculate` tool from the calculator extension.

In [ ]:
result = await agent.run("What is (2**10 - 24) * 3?")
print(result.content)

## 6. Self-Extension — Agent Creates a Tool

Ask the agent to create a new extension, then reload and verify.

In [ ]:
result = await agent.run(
    "Create a new extension file at extensions/greeter.py that registers a tool called 'greet'. "
    "The tool takes a 'name' parameter (string, required) and returns 'Hello, {name}! Welcome to ArcAgent.' "
    "Follow the same pattern as the calculator extension in extensions/calculator.py. "
    "Use the write tool to create the file."
)
print(result.content)

In [ ]:
# Verify the file was created
greeter_path = Path("workspace/extensions/greeter.py")
assert greeter_path.exists(), "greeter.py was not created!"
print("File created:")
print(greeter_path.read_text())

In [ ]:
# Hot reload to pick up the new extension
await agent.reload()

# Verify the greet tool is now registered
tools = agent._tool_registry.tools
print(f"Total tools after reload: {len(tools)}")
assert "greet" in tools, "greet tool not found after reload!"
print(f"\ngreet tool registered: source={tools['greet'].source}")

# List all extensions
for m in agent._extension_loader.manifests:
    print(f"  Extension: {m.name} -> tools={m.tools_registered}")

In [ ]:
# Use the new tool
result = await agent.run("Use the greet tool to greet Josh")
print(result.content)

## 7. Skill Creation

Agent creates a SKILL.md file, then re-discover skills.

In [ ]:
result = await agent.run(
    "Create a skill file at skills/code-review.md with this exact content:\n"
    "---\n"
    "name: code-review\n"
    "description: Review code for quality, security, and best practices\n"
    "version: 1.0.0\n"
    "requires: [read, grep, find]\n"
    "tags: [quality, security]\n"
    "category: development\n"
    "---\n\n"
    "# Code Review Skill\n\n"
    "When asked to review code:\n"
    "1. Read the target files\n"
    "2. Check for common issues\n"
    "3. Report findings\n"
)
print(result.content)

In [ ]:
# Reload to discover the new skill
await agent.reload()

print(f"Skills discovered: {len(agent.skills)}")
for skill in agent.skills:
    print(f"  {skill.name}: {skill.description}")
    print(f"    requires: {skill.requires}, tags: {skill.tags}")

## 8. Multi-Turn Chat with Memory

Test session persistence — agent remembers previous turns.

In [ ]:
# Turn 1
r1 = await agent.chat("My name is Josh and I'm building an agent framework called ArcAgent.")
print("Turn 1:", r1.content)

In [ ]:
# Turn 2 — references turn 1
r2 = await agent.chat("What's my name and what am I building?")
print("Turn 2:", r2.content)

# Verify memory
assert "Josh" in r2.content, "Agent didn't remember the name!"
assert "ArcAgent" in r2.content or "agent framework" in r2.content.lower(), "Agent didn't remember the project!"
print("\nMemory verified: agent remembers previous turns.")

In [ ]:
# Turn 3 — check session message count
session = agent._session
messages = session.get_messages()
print(f"Session ID: {session.session_id}")
print(f"Messages in session: {len(messages)}")
for msg in messages:
    role = msg.get('role', '?')
    content = msg.get('content', '')
    preview = content[:80] + '...' if len(content) > 80 else content
    print(f"  [{role}] {preview}")

## 9. ACE — Agentic Context Engineering

Agent writes to its own policy.md and context.md.

In [ ]:
result = await agent.run(
    "Write the following to policy.md: "
    "'Always explain your reasoning before taking action. "
    "When creating files, verify they exist after creation. "
    "Prefer using tools over guessing.'"
)
print(result.content)

# Verify
policy = Path("workspace/policy.md").read_text()
print(f"\npolicy.md content:\n{policy}")

In [ ]:
result = await agent.run(
    "Write the following to context.md: "
    "'Current project: ArcAgent demo. User: Josh. "
    "Key facts: 7 built-in tools, extensions auto-discovered from workspace/extensions/, "
    "skills discovered from workspace/skills/.'"
)
print(result.content)

# Verify
context = Path("workspace/context.md").read_text()
print(f"\ncontext.md content:\n{context}")

In [ ]:
# Now run a task — the system prompt should include both policy and context
result = await agent.run("What do you know about the current project and user?")
print(result.content)

## 10. Final Verification

Verify all workspace artifacts created during this demo.

In [ ]:
ws = Path("workspace")

checks = {
    "identity.md exists": (ws / "identity.md").exists(),
    "policy.md has content": len((ws / "policy.md").read_text().strip()) > 0,
    "context.md has content": len((ws / "context.md").read_text().strip()) > 0,
    "calculator extension": (ws / "extensions" / "calculator.py").exists(),
    "greeter extension": (ws / "extensions" / "greeter.py").exists(),
    "code-review skill": (ws / "skills" / "code-review.md").exists(),
    "8+ tools registered": len(agent._tool_registry.tools) >= 8,
    "greet tool available": "greet" in agent._tool_registry.tools,
    "1+ skills discovered": len(agent.skills) >= 1,
    "2+ extensions loaded": len(agent._extension_loader.manifests) >= 2,
}

print("Verification Results:")
print("=" * 40)
all_pass = True
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}] {check}")

print("=" * 40)
print(f"Overall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

## Cleanup

In [ ]:
await agent.shutdown()
print("Agent shut down.")